# Cascadia Cryptid Sighting Survey — the true model and five wrong ones

Ground truth lives in `simulate_cryptids.py`; the story and the full trap list are in `README.md`.
This notebook fits the correct model and then each shortcut, so the damage is visible side by side.

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf
from simulate_cryptids import simulate, TRUTH, DISTRICT_NAME

d = simulate()                                   # same seed as the shipped CSV
d["a_s"] = (d.canopy_vendor_a - 50) / 10         # vendor slopes apply to this scale
d["b_s"] = (d.canopy_vendor_b - 50) / 10
d["beer"] = d.beer_sales_z
d["logE"] = np.log(d.visitor_hundred_hours)      # the offset
d.head()

## Step 1 — is it Poisson? (no)

In [ ]:
print("variance / mean =", round(d.sightings.var() / d.sightings.mean(), 1))
print("zeros:", f"{(d.sightings == 0).mean():.1%}", " max:", d.sightings.max())
NB = lambda: sm.families.NegativeBinomial(alpha=1 / TRUTH["theta"])

## Step 2 — the correct model\nNB family, log-exposure offset, district as a factor, curvature in beer.

In [ ]:
FULL = "sightings ~ a_s + b_s + beer + I(beer**2) + C(ranger_district)"
true_fit = smf.glm(FULL, d, family=NB(), offset=d.logE).fit()
pd.DataFrame({"est": true_fit.params, "se": true_fit.bse, "p": true_fit.pvalues}).round(3)

## Step 3 — what the collinear vendor pair can and cannot tell us

At r = 0.974 only the common component is identified. The sum is estimated tightly;
the difference is not. The individually non-significant vendor coefficients above
are the honest answer, not a broken fit.

In [ ]:
def contrast(fit, terms):
    """Linear contrast of named coefficients: returns (estimate, se)."""
    c = np.zeros(len(fit.params))
    for name, weight in terms.items():
        c[list(fit.params.index).index(name)] = weight
    t = fit.t_test(c)
    return float(np.ravel(t.effect)[0]), float(np.ravel(t.sd)[0])

truth_sum = TRUTH["canopy_vendor_a"] + TRUTH["canopy_vendor_b"]
truth_diff = TRUTH["canopy_vendor_a"] - TRUTH["canopy_vendor_b"]
print("A + B: est %.3f se %.3f   (truth %.3f)" % (*contrast(true_fit, {"a_s": 1, "b_s": 1}), truth_sum))
print("A - B: est %.3f se %.3f   (truth %.3f)" % (*contrast(true_fit, {"a_s": 1, "b_s": -1}), truth_diff))
print("corr(A, B) = %.3f" % d.canopy_vendor_a.corr(d.canopy_vendor_b))

## Step 4 — the wrong models\nEach one is a single default decision away from the correct fit.

In [ ]:
def fit_table(formula, offset=None, family=None):
    """Fit a GLM and return estimates with p-values."""
    f = smf.glm(formula, d, family=family or NB(), offset=offset).fit()
    return f, pd.DataFrame({"est": f.params, "p": f.pvalues}).round(4)

pois, _ = fit_table(FULL, offset=d.logE, family=sm.families.Poisson())
print("TRAP 1 Poisson: chi2/df = %.1f, SEs are %.2fx the NB ones"
      % ((pois.resid_pearson ** 2).sum() / pois.df_resid, pois.bse["a_s"] / true_fit.bse["a_s"]))

SHORTCUT = "sightings ~ a_s + b_s + beer + ranger_district"   # no curvature, district numeric
_, no_off = fit_table(SHORTCUT)
print("\nTRAP 2 offset omitted (truth: A +0.330, B -0.295)")
print(no_off.loc[["a_s", "b_s"]])

_, with_off = fit_table(SHORTCUT, offset=d.logE)
print("\nTRAPS 3 and 4 offset restored, but district numeric and beer linear")
print(with_off.loc[["ranger_district", "beer"]])

drop_b, tab = fit_table("sightings ~ a_s + beer + I(beer**2) + C(ranger_district)", offset=d.logE)
print("\nTRAP 5 vendor B dropped to fix VIF: A = %.4f, p = %.4f (truth 0.330, identified sum %.3f)"
      % (drop_b.params["a_s"], drop_b.pvalues["a_s"], truth_sum))

## Step 5 — the test the numeric district coding throws away

In [ ]:
reduced = smf.glm("sightings ~ a_s + b_s + beer + I(beer**2)", d, family=NB(), offset=d.logE).fit()
print("district LR test: deviance drop %.1f on 4 df" % (reduced.deviance - true_fit.deviance))
for level, name in DISTRICT_NAME.items():
    print(f"  {level}  {name}")